<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/> 

# JWebbinar50: Introduction to Jdaviz 5.0, Part III

_July 7, 2026_

**Authors**: Sarah Betti (STScI), Matthew Portman (STScI), Ricky O'Steen (STScI)<br>
**Jdaviz Version**: 5.0.2

- [Documentation](https://jdaviz.readthedocs.io/en/latest/index.html)
- [Source code](https://github.com/spacetelescope/jdaviz)

## Tutorial Outline

This tutorial will demonstrate features recently introduced (since Spring 2026) to Jdaviz, with a focus on using the API Python calls for 2D and 1D spectra.  

This notebook will walk your though how to:
- create a `specutils.Spectrum` object of a 2D spectrum
- load in a `specutils.Spectrum` object into Jdaviz with auto extraction of a 1D spectrum
- load in a custom line list for spectral line analysis
- measure the line flux of each line
- create a new catalog and scatter plot from the results of the line analysis

The new features include the ability to:
- load custom list lists into the `Line Lists` plugin
- create a table or catalog in Jdaviz and to load it back into the application
- create new viewers



In [ ]:
import numpy as np

from astropy.io import fits
from astropy import units as u
from astropy.table import QTable

from specutils import Spectrum
from specutils import SpectralRegion

from astroquery.mast import Observations

import jdaviz as jd

## Set up our data

Here we load an example 2D spectra of the pre-main-sequence star with source ID 977 in NGC 3603 observed as part of a NIRSpec Guaranteed Time Observation program ID 1225.  The spectrum used here have been published in [Rogers et al., 2024](https://ui.adsabs.harvard.edu/abs/2024A%26A...684L...8R/abstract). We can download this file directly from the MAST archive using the astroquery package.

In [ ]:
filename = 'jw01225-o001_s000000977_nirspec_f170lp-g235h_s2d.fits'
uri = f'mast:JWST/product/{filename}'
_ = Observations.download_file(uri)

We then open our file with astropy and extract the WAVELENGTH HDU and the SCI HDU.  We find those are are located in the 1st and 3rd extension. By opening up the science header, we can determine the flux units as well. NIRSpec provides the wavelength array in microns.

In [ ]:
hdul = fits.open(filename)
hdul.info()

In [ ]:
wavelength = hdul[3].data[0]
flux = hdul[1].data

print('2D spectrum has units of ', hdul[1].header['BUNIT'])

Using specutils, we can create a Spectrum object, which can be read by Jdaviz.  We can specify our spectral axis and our 2D flux with units of microns and MJy, respectively.

In [ ]:
spec = Spectrum(spectral_axis = wavelength * u.um, flux = flux * u.MJy)

## Load the data and launch Jdaviz

Now, we initialize `jdaviz` and load in our Spectrum. The `format` keyword is required and tells Jdaviz what format to ingest the data as---in this case `2D Spectrum`. We additionally provide an optional `data_label` keyword to label our spectrum `ID 977`. Other options for `load` are given in the documentation.

As a reminder from Part I, if nothing pops open, make sure you allow pop ups on your browser, or change the cell using another options below.

Other options are:
- inline: Will appear below the jd.show(loc='inline') cell
- sidecar: Will appear to the right of the notebook.  This only works in jupyter lab environments. 

In [ ]:
jd.load(spec, format='2D Spectrum', data_label='ID 977')
jd.show(loc='popout:window')

We see 2 viewer windows: the top is our 2D spectrum and the bottom is the auto-extracted 1D spectrum.  If the auto extraction is not sufficient, you can re-extract using the `2D Spectral Extraction` plugin.  Documentation for how to do that is found [here](https://jdaviz.readthedocs.io/en/latest/plugins/2d_spectral_extraction.html), and an example tutorial is found [here](https://github.com/spacetelescope/jdat_notebooks/blob/main/notebooks/jdaviz_demo/notebooks/Specviz_Specviz2d.ipynb).  

Since this tutorial is aimed at only using the API, we can turn on the API hints. With them on, we can see the API calls as blue font. 

In [ ]:
jd.toggle_api_hints(enabled=True)

Each plugin has an API that can be accessed in code cells to interact with that plugin. To access the plugin you can run `jd.plugin['<plugin name'>]`

## Create and load a line list

Let's determine what our three emission lines are.  This spectrum is from a known accreting pre-main-sequence star. Therefore, the lines are likely Hydrogen emission lines. Between 1.7-3 microns, the brightest are Paschen alpha (1.875 um), Brackett beta (2.626 um), and Brackett gamma (2.165), of which Pa alpha and Br beta are only visible from space (This region of the spectrum is heavily absorbed by water vapor from Earth's atmosphere!)

Let's proceed by creating an astropy table of these lines to `load` into Jdaviz using `format='Line List'`. Doing so tells Jdaviz to port this table directly to the `Line Lists` plugin.

In [ ]:
hydrogen_line_list = QTable()
hydrogen_line_list['linename'] = ['Pa alpha', 'Br beta', 'Br gamma']
hydrogen_line_list['rest'] = [1.875, 2.626, 2.165] * u.um 
hydrogen_line_list

In [ ]:
# Load the custom line list into the Line Lists plugin
jd.load(hydrogen_line_list, format='Line List') 

We can now open the `Line Lists` plugin and access its API.

To see all the parameters and methods that are accessible through the API, you can run `dir(<Line List Plugin>)`.

In [ ]:
# Access the plugin API
linlist = jd.plugins['Line Lists']

# Running open_in_tray() opens the plugin tray in the UI.
# Note: This is NOT necessary to run API calls, but is useful to see the available features. 
linlist.open_in_tray()

dir(linlist)

Scrolling down we can see our line list under the `----Loaded Lines---` section in the `Imported` dropdown.  Selecting the dropdown allows us to see the lines. We can then select the PLOT ALL button to see the lines on the `1D Spectrum` viewer. 

They line up perfectly with our emission lines!  To see which is which, select the icon with the 3 vertical lines to the left of each eye icon. This should make the red line thicker in the viewer. 

## Create subsets

With our lines, we now want to measure the flux coming from the emission lines.  

To do this, we first need to define a subset around each line in the spectrum.  We use the `Subset Tools` plugin to define the region for each line.  

In [ ]:
st = jd.plugins['Subset Tools']

We use a specutils.SpectralRegion to create a region and the `Subset Tools` plugin call `import_region` to create the subset.  Finally, we rename each subset to specify the name of each line.  

In [ ]:
st.import_region(SpectralRegion(1.873 * u.um, 1.8774 * u.um))
st.rename_subset('Subset 1', 'Pa alpha')

In [ ]:
st.import_region(SpectralRegion(2.623 * u.um, 2.628 * u.um))
st.rename_subset('Subset 2', 'Br beta')

In [ ]:
st.import_region(SpectralRegion(2.164 * u.um, 2.167 * u.um))
st.rename_subset('Subset 3', 'Br gamma')

We can zoom in to one of the lines to see the subset using the `Plot Options` plugin.

In [ ]:
plo = jd.plugins['Plot Options']
plo.viewer = '1D Spectrum'
plo.x_min = 1.87
plo.x_max = 1.88

and reset back to the full spectrum

In [ ]:
plo.reset_viewer_bounds()

## Line Analysis

With our subsets, we can now measure the emission lines using the `Line Analysis` plugin. We first access the plugin and open it in the tray.  

In [ ]:
la = jd.plugins['Line Analysis']
la.open_in_tray()

Accessing the parameter associated with a selection menu will show both the choices and the current selection.

In [ ]:
print('dataset: ', la.dataset)
print('subsets: ', la.spectral_subset)

Here, we set the dataset to the only available choice. This is unnecessary in this case but when multiple choices are available, this is how you would change the selected dataset. 

In [ ]:
la.dataset = 'ID 977 (auto-ext)'

Next, we will set the spectral subset for the Pa alpha line, and set the continuum to use the `Surrounding` region with a width of `3` relative to the overall line spectral region. We then use `get_results` to compute the line statistics.

In [ ]:
la.spectral_subset  = 'Pa alpha'
la.continuum        = 'Surrounding'
la.continuum_width  = 3
la.get_results(add_to_table=True)

We then repeat this for our other two lines!  Only the properties that we update will change.

In [ ]:
la.spectral_subset  = 'Br beta'
la.get_results(add_to_table=True)

la.spectral_subset  = 'Br gamma'
la.get_results(add_to_table=True)

Finally, we export the table to the notebook where we can look at it. This table can also be seen in the `Line Analysis` plugin.  

In [ ]:
line_table = la.export_table()
line_table

## Loading the table into Jdaviz

There are two methods to load the table back into Jdaviz for analysis. The first is to use the UI directly. Underneath the table that we just created, there is a drop down, `Load Table into App`, where you can directly send the table back into the application.  

We can also load it back in using the API. To do so, we first access the loader plugin, and specify that we want to load an 'object'. You can consider this the object-oriented alternative to the `load` call we used previously for the 2D Spectrum. 

In [ ]:
ldr = jd.loaders['object']
ldr.open_in_tray()

We set our `object` to be the line table that we just exported.  

In [ ]:
ldr.object = line_table

Next, we specify the format as `Catalog`. This is the same as setting the format via `load` as `jd.load(object_name, format='Catalog')`. 

In [ ]:
ldr.format = 'Catalog'

We then tell the loader we want to create a new `Table` viewer with a label `Emission Line Analysis`. 

In [ ]:
ldr.importer.viewer.create_new = 'Table'
ldr.importer.viewer.new_label = 'Emission Line Analysis'

Finally, we set the x (`col_x`) and y (`col_y`) coordinate columns from the table, specify a few other columns of interest, and load the data back into Jdaviz.

In [ ]:
ldr.importer.col_x = 'Centroid'
ldr.importer.col_y = 'Line Flux'
ldr.importer.col_other = ['Line Flux','Line Flux:unit',
                          'Equivalent Width','Equivalent Width:unit',
                          'Centroid', 'Centroid:unit', 
                          'spectral_subset']
ldr.load()

You can now see a new viewer with the loaded table.  

## Creating and plotting a scatter plot

Now let's create another new viewer! Unlike previously, where the viewer was created when we loaded data into Jdaviz, we can create a new viewer and add the now-loaded table into it. We call `new_viewers` and create a `Scatter` viewer.

We again `open_in_tray()` to show where this feature is. We don't need to call `open_in_tray()` to perform the task in the API.

In [ ]:
vc = jd.new_viewers['Scatter']
vc.open_in_tray()

We set the name of our new viewer.

In [ ]:
vc.viewer_label = 'Line Flux vs Wavelength' 

We then select our newly loaded `Catalog`, and the x (`xatt`) and y (`yatt`) coordinate columns we want plotted. Finally, we call the API to create the plot.

In [ ]:
vc.dataset = 'Catalog'
vc.xatt = 'Centroid'
vc.yatt = 'Line Flux'

vc()

The scatter plot has been created!  You might also notice a red message that pops up in the plugin to indicate that the Viewer label is already in use. If you want to create another new viewer, you must use a different name. 

## Changing the Plot

Finally, we'll focus on adjusting the scatter plot by changing the size and color of the points to make them more prominent. We will go into the `Plot Options` plugin, select our scatter plot `Line Flux vs Wavelength` viewer, and change the marker size and marker colors. 

In [ ]:
po = jd.plugins['Plot Options']
po.open_in_tray()

po.viewer = 'Line Flux vs Wavelength'
po.layer.multiselect=False
po.layer = 'Catalog'
po.xatt = 'Centroid'
po.yatt = 'Line Flux'

po.marker_size_scale = 10
po.marker_size = 10
po.marker_color = '#F70202FF' #red

---
<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_footer.png' alt="stsci_logo" width="400px"/> 